# CT dose network follow-up (seed 42 only)

Applies the plan of `ctdosenet_latest.md` and nothing else. Weeks 1 and 2 are not rerun; their outputs are only read for comparison.

| Step | What | Output |
|---|---|---|
| A | Check the variance fix in `src/distances.py::local_stats` | gradient test |
| B | Recalibrate lambda, rerun **C8 (DINOv2-S, statistics)** and **C10 (CT dose network v1, statistics)** | `seed42/train_C8_varfix`, `seed42/train_C10_varfix_v1` |
| C | One retry of the dose network: log-dose regression target, validation checkpointing, judged on the **validation screen only** | `seed42/ct_dose_net_v2_regression`, `screen/*_val.csv`, `decision.json` |
| D | Only if v2 is adopted (M above 0.9 for both matchings): recalibrate and train **C9 (CT dose network v2, pointwise)** and **C10 (CT dose network v2, statistics)** | `seed42/train_C9_v2`, `seed42/train_C10_v2` |
| E | Summary tables | `ctdosenet_followup_tables.md` |

Seeds 123 and 456 are **not started** here. Nothing in this notebook pushes to git; commit and push `src/` yourself before running on Colab.

## 0. Run configuration
Edit this cell only.

In [ ]:
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

SEED = 42                           # the only seed of this notebook
SMOKE = False                       # True: 8 slices per split, 1 epoch everywhere (wiring check, not a result)

WORK_DIR = Path("/content") if IN_COLAB else Path.cwd()
REPO_DIR = Path.cwd() if (Path.cwd() / "src" / "config.py").exists() else WORK_DIR / "perceptual_backbone_ldct"
REPO_URL = "https://github.com/UnkindledTwo/perceptual_backbone_ldct"
DATA_ROOT = WORK_DIR / "minideeplesion"
DATASET_GDRIVE_ID = "1pIUT5beAkp17WCxyGPHZfHy4kEiJtD4y"
OUTPUT_DIR = WORK_DIR / "outputs" / "ctdosenet_followup"
DRIVE_DIR = Path("/content/drive/MyDrive/BIAI/ISBI2027_WEIGHTS") if IN_COLAB else None

# Earlier outputs, read only. The first folder that holds a file is used.
OLD_RESULTS_DIRS = [d for d in (DRIVE_DIR / "results" if DRIVE_DIR else None, REPO_DIR / "results") if d is not None]
# Frozen v1 dose network weights (gitignored, so they are not in the clone). Put the file in one of these places.
DOSE_V1_CANDIDATES = [p for p in (
    DRIVE_DIR / "ct_dose_net_v1.pt" if DRIVE_DIR else None,
    DRIVE_DIR / "ct_dose_net.pt" if DRIVE_DIR else None,
    REPO_DIR / "results" / "seed42" / "ct_dose_net_v1" / "ct_dose_net.pt",
    WORK_DIR / "ct_dose_net.pt",
) if p is not None]

# Protocol (paper_plan.md). Unchanged.
EPOCHS = 1 if SMOKE else 30
BATCH_SIZE = 8
CROP_SIZE = 224
TARGET_INFLUENCE = 0.95

# Dose network retry (Step C)
DOSE_TARGET_MODE = "regression"     # "ordinal" is the fallback if regression collapses; do not sweep
DOSE_NET_EPOCHS = 1 if SMOKE else 10
DOSE_PATCH_SIZE = 128
DOSE_PATCHES_PER_SLICE = 1 if SMOKE else 4
BODY_THRESHOLD = 0.05
MIN_BODY_FRACTION = 0.9
ADOPT_M_THRESHOLD = 0.9             # v2 replaces v1 only if M exceeds this for both matchings (validation screen)
FORCE_ADOPT_V2 = False              # True: adopt v2 although the rule is not met (recorded as an override in decision.json)

FORCE_RERUN = False
DETERMINISTIC = True
NUM_WORKERS = 0 if SMOKE else 4

## 1. Setup

In [ ]:
%pip install -q lpips piq huggingface_hub gdown

In [ ]:
if not DATA_ROOT.exists():
    !gdown {DATASET_GDRIVE_ID} -O {WORK_DIR}/dataset_biailab_I2I_2025v1.zip
    !unzip -q -o {WORK_DIR}/dataset_biailab_I2I_2025v1.zip -d {WORK_DIR}
if not (REPO_DIR / "src" / "config.py").exists():
    !git clone {REPO_URL} {REPO_DIR}
if IN_COLAB and DRIVE_DIR is not None:
    from google.colab import drive
    drive.mount("/content/drive")
assert DATA_ROOT.exists(), f"dataset not found at {DATA_ROOT}"
assert (REPO_DIR / "src" / "dose_net_train.py").exists(), (
    "src/dose_net_train.py is missing from the checkout: push the src/ changes (variance fix, dose_net_train.py) "
    "to GitHub and delete the stale clone, or upload src/ by hand")

In [ ]:
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import inspect
import json
import random
import shutil
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import Markdown, display
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

import src.distances as distances_module
from src.config import PerceptualConfig
from src.ct_dose_net import CTDoseNet, DoseNetFeatures
from src.data import I2IPairs, assert_no_patient_leakage, patient_id
from src.distances import FeatureDistance, PerceptualObjective
from src.dose_net_train import (DOSE_CLASS_NAMES, DosePatches, evaluate_dose_net, outputs_for, train_dose_net)
from src.feature_nets import build_feature_net
from src.metrics import slice_metrics
from src.model import UNet
from src.screen import DOSES, screen_network
from src.train import validation_l1

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RUN_ROOT = OUTPUT_DIR / f"seed{SEED}"
SCREEN_DIR = OUTPUT_DIR / "screen"
for folder in (OUTPUT_DIR, RUN_ROOT, SCREEN_DIR):
    folder.mkdir(parents=True, exist_ok=True)
print("device:", device, "| repo:", REPO_DIR, "| outputs:", OUTPUT_DIR, "| SMOKE:", SMOKE)
if device.type != "cuda":
    print("WARNING: no GPU. The 30-epoch runs of Steps B and D need one (Colab: Runtime > Change runtime type).")

In [ ]:
METRICS = ["psnr", "ssim", "lpips", "dists", "r_h"]


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = DETERMINISTIC
    torch.backends.cudnn.benchmark = not DETERMINISTIC


def _seed_worker(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % 2 ** 32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_loader(dataset: Dataset, batch_size: int, shuffle: bool = False, seed: int = SEED) -> DataLoader:
    """Loader whose shuffling and worker seeds come from its own generator, not the global RNG."""
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=NUM_WORKERS,
                      generator=generator, worker_init_fn=_seed_worker)


def sync() -> None:
    if device.type == "cuda":
        torch.cuda.synchronize()


def count_params(module: nn.Module | None) -> int:
    return 0 if module is None else sum(p.numel() for p in module.parameters())


def free(*modules: nn.Module | None) -> None:
    for module in modules:
        if module is not None:
            module.cpu()
    if device.type == "cuda":
        torch.cuda.empty_cache()


def find_old(*relative: str) -> Path | None:
    """First existing file among the read-only earlier result folders."""
    for folder in OLD_RESULTS_DIRS:
        if (folder.joinpath(*relative)).exists():
            return folder.joinpath(*relative)
    return None


def to_markdown(frame: pd.DataFrame, digits: int = 4) -> str:
    def cell(value) -> str:
        if isinstance(value, (bool, np.bool_)):
            return "yes" if value else "no"
        if isinstance(value, (float, np.floating)):
            return "-" if not np.isfinite(value) else f"{value:.{digits}f}"
        return "-" if value is None else str(value)

    lines = ["| " + " | ".join(map(str, frame.columns)) + " |", "|" + "---|" * len(frame.columns)]
    lines += ["| " + " | ".join(cell(v) for v in row) + " |" for row in frame.itertuples(index=False)]
    return "\n".join(lines)

In [ ]:
FILES = {split: sorted((DATA_ROOT / split).glob("*.npz")) for split in ("train", "val", "test")}
patients_per_split = assert_no_patient_leakage(DATA_ROOT)
print("patients per split:", patients_per_split, "| slices:", {s: len(f) for s, f in FILES.items()})
DATASETS = {split: I2IPairs(DATA_ROOT / split, "noisy_clipped_100k", "gt_clipped") for split in ("train", "val", "test")}
if SMOKE:
    for split in FILES:
        FILES[split] = FILES[split][:8]
        DATASETS[split].files = FILES[split]
else:
    assert {s: len(f) for s, f in FILES.items()} == {"train": 899, "val": 182, "test": 220}, "split sizes differ from the Protocol"

## Step A: variance fix check

`local_stats` must use `(variance.clamp(min=0) + eps).sqrt()`. The old `clamp(min=1e-8)` zeroed the gradient of the local standard deviation in flat windows, which a screen cannot show (it changes distance values by about 1e-4). The check below inspects the source and runs a gradient test.

In [ ]:
source = inspect.getsource(distances_module.local_stats)
assert "clamp(min=0)" in source and "variance + eps" in source, "src/distances.py still has the old variance clamp: push the fix"

flat = torch.full((1, 4, 16, 16), 0.3, requires_grad=True)
flat_stats = distances_module.local_stats(flat, 4)
flat_stats[1].sum().backward()
assert torch.isfinite(flat.grad).all(), "NaN or inf gradient in a flat window"

rng = torch.Generator().manual_seed(0)
near_flat = (0.3 + 5e-5 * torch.randn(1, 4, 16, 16, generator=rng)).requires_grad_(True)   # variance about 2.5e-9, below the old floor
distances_module.local_stats(near_flat, 4)[1].sum().backward()
print(f"std gradient, near-flat window: nonzero in {(near_flat.grad != 0).float().mean():.1%} of entries "
      f"(old clamp: 0%), finite: {bool(torch.isfinite(near_flat.grad).all())}")

# End to end: a near-flat prediction (as early in training) against a textured target.
textured = torch.rand(1, 4, 16, 16, generator=rng)
prediction = (0.3 + 5e-5 * torch.randn(1, 4, 16, 16, generator=rng)).requires_grad_(True)
sum(distances_module.statistics_blocks([prediction], [textured])).backward()
finite, nonzero = bool(torch.isfinite(prediction.grad).all()), float((prediction.grad != 0).float().mean())
print(f"statistics distance, near-flat prediction: finite {finite}, nonzero in {nonzero:.1%} of entries")
assert finite, "NaN or inf gradient in the statistics distance"
assert nonzero > 0, "statistics distance gives no gradient for a near-flat prediction"
print("variance fix present")

## Step B: recalibrate lambda, rerun C8 and C10 (seed 42)

Same rule as Step 2.1: lambda such that the perceptual term is 95% of the initial loss, from the same initial U-Net theta_0, the same random crops and the same training split. The other configurations keep their lambda. Runs go to new folders; the earlier C8 and C10 outputs are not touched.

In [ ]:
def load_dose_net(path: Path, outputs: int) -> CTDoseNet:
    network = CTDoseNet(outputs)
    network.load_state_dict(torch.load(path, map_location="cpu"))
    return network


def make_distance(name: str, matching: str, dose_weights: Path | None = None, dose_outputs: int = 6) -> FeatureDistance:
    if name == "ct_dose_net":
        net = DoseNetFeatures(load_dose_net(dose_weights, dose_outputs))
    else:
        net = build_feature_net(name)
    return FeatureDistance(net, matching)


def random_crop_pair(prediction: torch.Tensor, target: torch.Tensor, size: int, generator: torch.Generator):
    top = torch.randint(0, prediction.shape[-2] - size + 1, (1,), generator=generator).item()
    left = torch.randint(0, prediction.shape[-1] - size + 1, (1,), generator=generator).item()
    return prediction[..., top:top + size, left:left + size], target[..., top:top + size, left:left + size]


@torch.no_grad()
def initial_loss_terms(model: nn.Module, distance: nn.Module, seed: int = SEED) -> tuple[float, float]:
    """Mean L1 term and mean feature distance over the training split at initialization."""
    crop_rng = torch.Generator().manual_seed(seed)
    pixel_sum, feature_sum, seen = 0.0, 0.0, 0
    for noisy, clean, _ in tqdm(make_loader(DATASETS["train"], BATCH_SIZE), desc="calibrate", leave=False):
        prediction, clean = model(noisy.to(device)).float(), clean.to(device)
        pixel_sum += F.l1_loss(prediction, clean).item() * noisy.size(0)
        feature_sum += distance(*random_crop_pair(prediction, clean, CROP_SIZE, crop_rng)).item() * noisy.size(0)
        seen += noisy.size(0)
    return pixel_sum / seen, feature_sum / seen


seed_everything(SEED)
initial_model = UNet().to(device).eval()        # theta_0, identical to the one behind every earlier row


def calibrate(distance: nn.Module) -> dict:
    s_1, s_phi = initial_loss_terms(initial_model, distance)
    weight = TARGET_INFLUENCE / (1 - TARGET_INFLUENCE) * s_1 / s_phi
    return {"S_1": s_1, "S_phi": s_phi, "lambda": weight, "influence": weight * s_phi / (s_1 + weight * s_phi)}


old_path = find_old("calibration_weights.csv")
OLD_CALIBRATION = pd.read_csv(old_path).set_index("Configuration") if old_path else None
print("earlier calibration table:", old_path)

In [ ]:
METRIC_COLUMNS = METRICS
WARMUP_STEPS = 10


def train_run(run_name: str, config_id: str, label: str, feature_net: str, matching: str, loss_weight: float,
              distance_factory, seed: int = SEED) -> tuple[pd.DataFrame, dict]:
    """Protocol training loop of the main notebook (timed copy of src.train.train_and_test), output to RUN_ROOT/run_name.

    `distance_factory` builds the feature distance after the U-Net, so the initial U-Net and the crop sequence match every earlier run.
    """
    config = PerceptualConfig(run_name=run_name, data_root=DATA_ROOT, feature_net=feature_net, matching=matching,
                              loss_weight=float(loss_weight), output_dir=RUN_ROOT, crop_size=CROP_SIZE, seed=seed,
                              batch_size=BATCH_SIZE, epochs=EPOCHS)
    run_dir = config.run_dir
    run_dir.mkdir(parents=True, exist_ok=True)
    summary_path, metrics_path, checkpoint = run_dir / "run_summary.json", run_dir / "metrics.csv", run_dir / "best.pth"
    if summary_path.exists() and metrics_path.exists() and not FORCE_RERUN:
        print(f"{label}: finished run found in {run_dir}, loading it")
        return pd.read_csv(metrics_path, dtype={"patient_id": str}), json.loads(summary_path.read_text())

    seed_everything(seed)
    model = UNet().to(device)
    distance = distance_factory()
    criterion = PerceptualObjective(distance, config.loss_weight, config.crop_size).to(device)
    seed_everything(seed)

    train_loader = make_loader(DATASETS["train"], config.batch_size, shuffle=True, seed=seed)
    val_loader = make_loader(DATASETS["val"], config.batch_size)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5)
    use_amp = config.use_amp and device.type == "cuda"
    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    best, best_epoch, nonfinite, step_ms, log = float("inf"), -1, 0, [], []
    run_start = time.perf_counter()
    for epoch in tqdm(range(config.epochs), desc=run_name):
        model.train()
        epoch_start, running, seen = time.perf_counter(), 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        for step, (noisy, clean, _) in enumerate(train_loader):
            noisy, clean = noisy.to(device), clean.to(device)
            sync()
            step_start = time.perf_counter()
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                prediction = model(noisy)
            loss = criterion(prediction.float(), clean)       # perceptual term in float32
            if not torch.isfinite(loss):
                nonfinite += 1
                optimizer.zero_grad(set_to_none=True)
                continue
            scaler.scale(loss / config.grad_accumulation).backward()
            if (step + 1) % config.grad_accumulation == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
            sync()
            step_ms.append((time.perf_counter() - step_start) * 1e3)
            running += loss.item() * noisy.size(0)
            seen += noisy.size(0)
        score = validation_l1(model, val_loader, device)
        scheduler.step(score)
        if score < best:
            best, best_epoch = score, epoch + 1
            torch.save(model.state_dict(), checkpoint)
        log.append({"epoch": epoch + 1, "train_loss": running / max(seen, 1), "val_l1": score,
                    "lr": optimizer.param_groups[0]["lr"], "epoch_seconds": time.perf_counter() - epoch_start})
    train_seconds = time.perf_counter() - run_start
    pd.DataFrame(log).to_csv(run_dir / "train_log.csv", index=False)

    model.load_state_dict(torch.load(checkpoint, map_location=device))
    model.eval()
    rows, test_start = [], time.perf_counter()
    with torch.no_grad():
        for noisy, clean, stem in DATASETS["test"]:           # the test split is touched once, here
            prediction = model(noisy[None].to(device)).clamp(0, 1)[0, 0].float().cpu().numpy()
            rows.append({"slice_id": stem, "patient_id": patient_id(Path(stem)),
                         **slice_metrics(prediction, clean[0].numpy())})
    test_seconds = time.perf_counter() - test_start
    frame = pd.DataFrame(rows)
    frame.to_csv(metrics_path, index=False)

    timed = step_ms[WARMUP_STEPS:] if len(step_ms) > WARMUP_STEPS else step_ms
    summary = {
        "config": config_id, "label": label, "feature_net": feature_net, "matching": matching,
        "loss_weight": config.loss_weight, "seed": seed, "epochs": config.epochs, "batch_size": config.batch_size,
        "ms_per_step": float(np.mean(timed)) if timed else float("nan"),
        "steps": len(step_ms), "nonfinite_steps": nonfinite,
        "train_seconds": train_seconds, "test_seconds": test_seconds, "gpu_hours": (train_seconds + test_seconds) / 3600,
        "best_val_l1": best, "best_epoch": best_epoch,
        "unet_parameters": count_params(model), "feature_parameters": count_params(distance), "n_test": len(frame),
        **{f"{m}_mean": float(frame[m].mean()) for m in METRICS},
        **{f"{m}_std": float(frame[m].std(ddof=1)) for m in METRICS},
    }
    summary_path.write_text(json.dumps(summary, indent=2))
    free(model, criterion)
    print(f"{label}: PSNR {summary['psnr_mean']:.2f} dB, R_H {summary['r_h_mean']:.3f}, "
          f"{summary['ms_per_step']:.0f} ms/step, {summary['gpu_hours']:.2f} GPU h, {nonfinite} non-finite steps")
    return frame, summary

In [ ]:
DOSE_V1_PATH = next((p for p in DOSE_V1_CANDIDATES if p.exists()), None)
assert DOSE_V1_PATH is not None, "ct_dose_net.pt (v1) not found. Upload it to one of:\n  " + "\n  ".join(map(str, DOSE_V1_CANDIDATES))
print("v1 weights:", DOSE_V1_PATH)

STEP_B = {
    "C8": dict(run_name="train_C8_varfix", label="C8 (DINOv2-S, statistics)", net="dinov2_s", matching="statistics",
               dose_weights=None, dose_outputs=6, old_run="train_C8"),
    "C10": dict(run_name="train_C10_varfix_v1", label="C10 (CT dose network v1, statistics)", net="ct_dose_net",
                matching="statistics", dose_weights=DOSE_V1_PATH, dose_outputs=6, old_run="train_C10_v1"),
}
calibration_rows, WEIGHTS_B = [], {}
for config_id, spec in STEP_B.items():
    distance = make_distance(spec["net"], spec["matching"], spec["dose_weights"], spec["dose_outputs"]).to(device)
    row = calibrate(distance)
    free(distance)
    old = OLD_CALIBRATION.loc[config_id] if OLD_CALIBRATION is not None and config_id in OLD_CALIBRATION.index else None
    calibration_rows.append({"Configuration": spec["label"], "S_1": row["S_1"], "S_phi": row["S_phi"],
                             "lambda_new": row["lambda"], "influence": row["influence"],
                             "lambda_old": float(old["lambda"]) if old is not None else float("nan")})
    WEIGHTS_B[config_id] = row["lambda"]
df_calibration_b = pd.DataFrame(calibration_rows)
df_calibration_b.to_csv(OUTPUT_DIR / "calibration_varfix.csv", index=False)
if OLD_CALIBRATION is not None and not SMOKE:
    drift = abs(df_calibration_b["S_1"].iloc[0] / float(OLD_CALIBRATION.loc["C8", "S_1"]) - 1)
    print(f"S_1 against the earlier calibration: relative difference {drift:.2e} (same theta_0 and crops expected)")
display(df_calibration_b)

In [ ]:
RESULTS_B, SUMMARIES_B = {}, {}
for config_id, spec in STEP_B.items():
    print(f"\n>>> {spec['label']} (lambda {WEIGHTS_B[config_id]:.4f})")
    RESULTS_B[config_id], SUMMARIES_B[config_id] = train_run(
        spec["run_name"], config_id, spec["label"], spec["net"], spec["matching"], WEIGHTS_B[config_id],
        lambda spec=spec: make_distance(spec["net"], spec["matching"], spec["dose_weights"], spec["dose_outputs"]))

In [ ]:
rows = []
for config_id, spec in STEP_B.items():
    old_metrics = find_old("seed42", spec["old_run"], "metrics.csv")
    old_summary = find_old("seed42", spec["old_run"], "run_summary.json")
    for tag, frame, summary in (
            ("before (clamp 1e-8)", pd.read_csv(old_metrics) if old_metrics else None,
             json.loads(old_summary.read_text()) if old_summary else None),
            ("after (variance fix)", RESULTS_B[config_id], SUMMARIES_B[config_id])):
        if frame is None:
            rows.append({"Configuration": spec["label"], "Run": tag + ": earlier output not found"})
            continue
        rows.append({"Configuration": spec["label"], "Run": tag, "lambda": summary["loss_weight"],
                     **{m.upper(): f"{frame[m].mean():.4f} ± {frame[m].std(ddof=1):.4f}" for m in METRICS},
                     "best epoch": summary.get("best_epoch"), "non-finite steps": summary.get("nonfinite_steps")})
df_before_after = pd.DataFrame(rows)
df_before_after.to_csv(OUTPUT_DIR / "step_b_before_after.csv", index=False)
display(df_before_after)

## Step C: dose network v2 (one time-boxed retry)

Only the training target changes: same three-block feature extractor, same patches (same seeds, 128 x 128, at least 90% body, patient-disjoint train and validation), same optimiser. The 6-way cross-entropy is replaced by a **log-dose regression** (clean is placed at log10 = 7) and the epoch with the lowest validation loss is kept. The result is judged on the **validation screen** (centre crop, 182 slices) and nothing else; the test split is not used here.

Decision rule: v2 is adopted if M exceeds `ADOPT_M_THRESHOLD` for both matchings. Otherwise v1 is frozen and reported as it is.

In [ ]:
DOSE_V2_DIR = RUN_ROOT / f"ct_dose_net_v2_{DOSE_TARGET_MODE}"
DOSE_V2_PATH = DOSE_V2_DIR / "ct_dose_net.pt"
DOSE_V2_DIR.mkdir(parents=True, exist_ok=True)

train_patches = DosePatches(FILES["train"], seed=SEED, patch=DOSE_PATCH_SIZE, per_slice=DOSE_PATCHES_PER_SLICE,
                            threshold=BODY_THRESHOLD, min_fraction=MIN_BODY_FRACTION)
val_patches = DosePatches(FILES["val"], seed=SEED + 1, patch=DOSE_PATCH_SIZE, per_slice=DOSE_PATCHES_PER_SLICE,
                          threshold=BODY_THRESHOLD, min_fraction=MIN_BODY_FRACTION)
train_patients = {patient_id(p) for p in FILES["train"]}
val_patients = {patient_id(p) for p in FILES["val"]}
assert not train_patients & val_patients, "patient leakage between dose-network splits"
print(f"train patches {len(train_patches)} | val patches {len(val_patches)} | mean body fraction "
      f"{np.mean(train_patches.fractions):.3f} / {np.mean(val_patches.fractions):.3f}")
if not SMOKE:
    assert (len(train_patches), len(val_patches)) == (21576, 4368), "patch counts differ from v1"

In [ ]:
report_path = DOSE_V2_DIR / "report.json"
if DOSE_V2_PATH.exists() and report_path.exists() and not FORCE_RERUN:
    dose_report = json.loads(report_path.read_text())
    print("dose network v2: finished run found, loading it")
else:
    seed_everything(SEED)
    network = CTDoseNet(outputs_for(DOSE_TARGET_MODE)).to(device)
    start = time.perf_counter()
    dose_log = train_dose_net(network, make_loader(train_patches, 64, shuffle=True, seed=SEED),
                              make_loader(val_patches, 64), device, mode=DOSE_TARGET_MODE,
                              epochs=DOSE_NET_EPOCHS, checkpoint=DOSE_V2_PATH)
    seconds = time.perf_counter() - start
    pd.DataFrame(dose_log).to_csv(DOSE_V2_DIR / "train_log.csv", index=False)
    best = evaluate_dose_net(network, make_loader(val_patches, 64), device, DOSE_TARGET_MODE)
    confusion = best.pop("confusion")
    dose_report = {**best, "mode": DOSE_TARGET_MODE, "epochs": DOSE_NET_EPOCHS,
                   "best_epoch": int(min(dose_log, key=lambda r: r["val_loss"])["epoch"]),
                   "train_seconds": seconds, "gpu_hours": seconds / 3600,
                   "parameters": count_params(network), "feature_extractor_parameters": count_params(network.blocks),
                   "train_patches": len(train_patches), "val_patches": len(val_patches)}
    report_path.write_text(json.dumps(dose_report, indent=2))
    pd.DataFrame(confusion, index=DOSE_CLASS_NAMES, columns=DOSE_CLASS_NAMES).to_csv(DOSE_V2_DIR / "confusion_matrix.csv")
    free(network)

v1_report_path = find_old("seed42", "ct_dose_net_v1", "report.json")
v1_report = json.loads(v1_report_path.read_text()) if v1_report_path else {}
display(pd.DataFrame({"v1 (cross-entropy)": {k: v1_report.get(k) for k in ("val_accuracy", "val_accuracy_clean_vs_1000k", "epochs")},
                      f"v2 ({DOSE_TARGET_MODE})": {k: dose_report.get(k) for k in ("val_accuracy", "val_accuracy_clean_vs_1000k", "epochs")}}))
print(f"v2: validation MAE {dose_report['val_mae_log10']:.3f} log10 units, best epoch {dose_report['best_epoch']}")

In [ ]:
def screen_summary(frame: pd.DataFrame) -> dict:
    return {"ordering_M": frame["ordering"].mean(),
            "dose_ratio_10k_1000k": frame["d_10000"].mean() / frame["d_1000000"].mean(),
            "blur_penalty_B": frame["d_blur"].sum() / frame["d_100000"].sum()}


screen_rows = []
for tag, weights, outputs in (("v1", DOSE_V1_PATH, 6), (f"v2_{DOSE_TARGET_MODE}", DOSE_V2_PATH, outputs_for(DOSE_TARGET_MODE))):
    for matching in ("pointwise", "statistics"):
        distance = make_distance("ct_dose_net", matching, weights, outputs).to(device)
        frame = screen_network(distance, FILES["val"], device, crop=CROP_SIZE)
        frame.to_csv(SCREEN_DIR / f"ct_dose_net_{matching}_{tag}_val.csv", index=False)
        screen_rows.append({"Dose network": tag, "Matching": matching, **screen_summary(frame)})
        free(distance)
df_dose_screen = pd.DataFrame(screen_rows)
df_dose_screen.to_csv(OUTPUT_DIR / "dose_net_validation_screen.csv", index=False)
display(df_dose_screen)

In [ ]:
v2_tag = f"v2_{DOSE_TARGET_MODE}"
m_v2 = df_dose_screen[df_dose_screen["Dose network"] == v2_tag].set_index("Matching")["ordering_M"]
RULE_MET = bool((m_v2 > ADOPT_M_THRESHOLD).all())
ADOPT_V2 = RULE_MET or FORCE_ADOPT_V2
decision = {"adopt_v2": ADOPT_V2, "rule_met": RULE_MET, "override": ADOPT_V2 and not RULE_MET,
            "rule": f"M(v2) > {ADOPT_M_THRESHOLD} for pointwise and statistics (validation screen)",
            "M_v2": m_v2.to_dict(),
            "M_v1": df_dose_screen[df_dose_screen["Dose network"] == "v1"].set_index("Matching")["ordering_M"].to_dict(),
            "mode": DOSE_TARGET_MODE, "smoke": SMOKE}
(OUTPUT_DIR / "decision.json").write_text(json.dumps(decision, indent=2))
if ADOPT_V2 and not RULE_MET:
    print("OVERRIDE: the adoption rule is NOT met; v2 is adopted by decision. Report this in the paper.")
print("ADOPT v2: C9 and C10 are retrained on it (Step D)." if ADOPT_V2 else
      "FREEZE v1: C9 and C10 stay on v1 (C10 is the Step B rerun). Report v1 with its diagnosis, v2 as a disclosed negative result.")
print(decision)

## Step D: C9 and C10 on dose network v2 (only if adopted)

In [ ]:
STEP_D = {
    "C9": dict(run_name="train_C9_v2", label=f"C9 (CT dose network v2 {DOSE_TARGET_MODE}, pointwise)", matching="pointwise"),
    "C10": dict(run_name="train_C10_v2", label=f"C10 (CT dose network v2 {DOSE_TARGET_MODE}, statistics)", matching="statistics"),
}
RESULTS_D, SUMMARIES_D, rows_d = {}, {}, []
if not ADOPT_V2:
    print("v2 not adopted: nothing to train in this step.")
else:
    outputs_v2 = outputs_for(DOSE_TARGET_MODE)
    for config_id, spec in STEP_D.items():
        distance = make_distance("ct_dose_net", spec["matching"], DOSE_V2_PATH, outputs_v2).to(device)
        row = calibrate(distance)
        free(distance)
        spec["lambda"] = row["lambda"]
        rows_d.append({"Configuration": spec["label"], **row})
    display(pd.DataFrame(rows_d))
    pd.DataFrame(rows_d).to_csv(OUTPUT_DIR / "calibration_v2.csv", index=False)
    for config_id, spec in STEP_D.items():
        print(f"\n>>> {spec['label']} (lambda {spec['lambda']:.4f})")
        RESULTS_D[config_id], SUMMARIES_D[config_id] = train_run(
            spec["run_name"], config_id, spec["label"], "ct_dose_net", spec["matching"], spec["lambda"],
            lambda spec=spec: make_distance("ct_dose_net", spec["matching"], DOSE_V2_PATH, outputs_v2))

## Step E: summary tables

In [ ]:
free(initial_model)
blocks = ["## Step B: C8 and C10 after the variance fix (seed 42, test split)", to_markdown(df_before_after),
          "", "### Lambda", to_markdown(df_calibration_b, 4),
          "", "## Step C: dose network, validation screen and training", to_markdown(df_dose_screen),
          "", f"Decision: {'ADOPT v2' if ADOPT_V2 else 'FREEZE v1'} (rule: {decision['rule']}; rule met: {RULE_MET}"
          f"{', adopted by override' if decision['override'] else ''}).",
          "", f"v2 training: validation accuracy {dose_report['val_accuracy']:.2%}, clean vs 1000k "
              f"{dose_report['val_accuracy_clean_vs_1000k']:.2%}, MAE {dose_report['val_mae_log10']:.3f} log10 units, "
              f"best epoch {dose_report['best_epoch']} of {dose_report['epochs']}."]
if ADOPT_V2:
    rows = []
    for config_id, spec in STEP_D.items():
        frame, summary = RESULTS_D[config_id], SUMMARIES_D[config_id]
        rows.append({"Configuration": spec["label"], "lambda": summary["loss_weight"],
                     **{m.upper(): f"{frame[m].mean():.4f} ± {frame[m].std(ddof=1):.4f}" for m in METRICS}})
    for config_id, run in (("C9", "train_C9_v1"), ("C10", "train_C10_varfix_v1")):
        path = find_old("seed42", run, "metrics.csv") or (RUN_ROOT / run / "metrics.csv")
        if path.exists():
            frame = pd.read_csv(path)
            rows.append({"Configuration": f"{config_id} (CT dose network v1, {'pointwise' if config_id == 'C9' else 'statistics'}), disclosed ablation",
                         **{m.upper(): f"{frame[m].mean():.4f} ± {frame[m].std(ddof=1):.4f}" for m in METRICS}})
    blocks += ["", "## Step D: C9 and C10 on v2 against v1", to_markdown(pd.DataFrame(rows))]
(OUTPUT_DIR / "ctdosenet_followup_tables.md").write_text("\n".join(blocks))
display(Markdown("\n".join(blocks)))

In [ ]:
if DRIVE_DIR is not None and DRIVE_DIR.parent.exists():
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    archive = shutil.make_archive(str(DRIVE_DIR / "outputs_ctdosenet_followup"), "zip", str(OUTPUT_DIR))
    print("backup written to", archive)
else:
    print("no Drive folder configured; outputs stay in", OUTPUT_DIR)

Seeds 123 and 456 were **not** run by this notebook. They start separately (target: Tue Oct 13), using the dose network chosen in Step C. Nothing here touched git.